# Quantum Circuit Compilation from Source

This notebook builds circuits for the three benchmarks of the competition: an inverse-square-root phase for Coulomb interactions in fusion plasmas, a Reed–Solomon decoder for Decoded Quantum Interferometry (DQI), and the angle lookups of a FeMoCo block encoding. It then checks each circuit with the official verifier at its full number of shots. All circuits are kickmix circuits, made of reversible classical gates, phase gates and measurement-based uncomputation, so a classical simulator can check them input by input.

The score is the geometric mean of $\text{cost}_\text{ref}/\text{cost}$ over the three tasks. The costs are defined by the organizers as follows.

| Task | Cost | Reference circuit | This notebook |
|---|---|---|---|
| Inverse square root | Q × (Toffoli + T / 4) | Q 260, 7,735 Toffolis | Q 68, 1,768 Toffoli-equivalents |
| DQI decoder | R(ε₊) × Q² × T | Q 1,893, 5.73 M Toffolis | Q 881, 0.96 M Toffolis |
| FeMoCo lookups | λ_eff × T_total × Q_total | Q 730, 93,237 Toffolis, λ_eff 49.30 | Q 448, 23,500 Toffolis, λ_eff 48.75 |

Q is the qubit count. In the first row T counts T gates, and synthesized rotations are priced in T gates. In the second row T is the mean Toffoli count per shot, and R(ε₊) grows with the decoder's measured failure rate. T_total and Q_total cover the whole block encoding, including the organizers' state-preparation costs.

Each section below follows the same order: why the subroutine matters, what the circuit computes, where the savings come from, and which approximation is used and how much of the accuracy budget it spends.

In [ ]:
import json, math, os, pathlib, sys
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get('QCC_INPUT', '/kaggle/input'))
WORK = pathlib.Path(os.environ.get('QCC_WORK', '/kaggle/working'))
SUB = pathlib.Path(os.environ.get('QCC_SUB', '/tmp/qcc_submission'))
SLUG = 'google-verifiable-quantum-circuit-compilation'
COMP = next(p for p in [INPUT / 'competitions' / SLUG, INPUT / SLUG] if (p / 'kickmix').exists())
# The chemistry numeric input is located by the build cell.
WORKERS = int(os.environ.get('QCC_WORKERS', os.cpu_count() or 4))
for d in (SUB, WORK / 'src' / 'gen_invsqrt_r2', WORK / 'src' / 'gen_dqi_v6' / 'ref'):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439474177519.598, 'dfthcsos': 3355599588.3155365}

for _d in []:
    (WORK / _d).mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib, shutil
def sha(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
OWN = next(p.parent for p in INPUT.rglob('femoco_thc_r5.npz'))
shutil.copytree(next(OWN.rglob('public_v11.py')).parent, WORK / 'src', dirs_exist_ok=True)
OPEN = OWN / 'open_circuits'
PINS = {
    'invsqrt.kmb': '7bde83f48462a245b9307fe0d9db985d9b2e9b5b1901a48b21df96eca2106985',
    'dfthcsos/angle_k_0.kmb': '8b0771aea947c282c0da204b59366b01bf8c8ac8e20bb6e36bb9cc645af6d04f',
    'dfthcsos/angle_k_1.kmb': 'dac3cbc0b560c6e1d4d2787751baf2a372d1d49161ecd7101e3011c02c753d37',
    'dfthcsos/angle_k_10.kmb': 'a027d89d3f522ff373270196c1ed56975b97cc820490f4e061545e24ea990eb4',
    'dfthcsos/angle_k_11.kmb': 'a07301189a00a4c85c43ea6a3e42242bce229d65059912823ecf70459ab73b4d',
    'dfthcsos/angle_k_12.kmb': 'dc66d18e5c84624b37b2a4a1f5388054b1c395299848fed6291f9b5dfc4aa6c0',
    'dfthcsos/angle_k_13.kmb': '2ba7002934812b0e80f927cbfb47f03bd5b8ea86042b744dfb3b349f508de274',
    'dfthcsos/angle_k_14.kmb': '102bf40e6e1b8dca8e640cca669f530f73cf185c79a1979b60147503003f5ae5',
    'dfthcsos/angle_k_2.kmb': '48338c0ed7047be27f526fc4c8ee10f117428cd239a0ce8db6c1a817244d6ed9',
    'dfthcsos/angle_k_3.kmb': '62d53a0c8e2f64b9336c118b75adf6d77c569d0b6dde77166a87681e54cf1305',
    'dfthcsos/angle_k_4.kmb': 'bdebd22ed26449aceacd44ec262a47555ef6d064f945c3d4e3859126c1df704b',
    'dfthcsos/angle_k_5.kmb': 'e3de4485c276b71b50878cf2af1c9a93b78f487670b6a539219bd609939047a0',
    'dfthcsos/angle_k_6.kmb': '2d8de690f8b09b906c450cd7aad325fc135e33fdbc63263956a48126d4175445',
    'dfthcsos/angle_k_7.kmb': 'a784b6b89f8a6a3b6813f92e18f49037cda630bbc9d73c2cfb621fb0b0586deb',
    'dfthcsos/angle_k_8.kmb': '0de383e2cadf1b4465193e61a3aab43210eb5207e2fe0c92bdbde490653ba635',
    'dfthcsos/angle_k_9.kmb': '88505bbef8cd3b49e5d96d9264ebf254bc78d678523b6d08709f0c5a103fce7a',
    'dfthcsos/submission.npz': '29bebc6e3aab18e758985b30c5e0ca8a1b77c4b0dfd55caddc2f9ee2046fa79e',
}
sys.path.insert(0, str(WORK / 'src'))
import qcc_pipeline as P
P.setup(COMP)
import public_v11 as V
V.build_dqi(SUB, WORK / 'build' / 'dqi', COMP)
for name, pin in PINS.items():
    assert sha(OPEN / name) == pin, name
    (SUB / name).parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(OPEN / name, SUB / name)
REFIT = OWN / 'femoco_r4_b303_refit.npz'
assert sha(REFIT) == '35bda9555d76f74430ef074f08f2dcdc24686305a630a489fff77ba9a283a55b'
shutil.copyfile(REFIT, SUB / 'dfthcsos' / 'submission.npz')
RESULTS = P.verify_all(SUB, 'notebook-verification', workers=WORKERS)
assert all(r['status'] == 'PASS' for r in RESULTS.values())
(WORK / 'verification.json').write_text(json.dumps(RESULTS, indent=1, default=str))
P.write_zip(SUB, WORK / 'submission.zip')

## 1. Inverse square root

**Role.** Stopping-power simulations of warm dense plasmas evaluate a pairwise Coulomb potential. The quantum subroutine applies the phase $e^{2\pi i/\sqrt{x}}$ to a 22-bit input $x$. Every gate in it is repeated for every particle pair and time step.

**Computation.** Binary shift stages normalize $x$ to a mantissa $M \in [1, 2)$. Their flags give the shift, with $e$ its parity and $s$ half of it:

$$\frac{1}{\sqrt{x}} = 2^{\,s-10}\, y, \qquad y = \frac{2^{(e-1)/2}}{\sqrt{M}} \in (0.5, 1].$$

A lookup addressed by $e$ and the high mantissa bits loads the coefficients of a centered quadratic. The low bits form an offset whose digits are all $\pm 1$:

$$\delta = \sum_i (2d_i - 1)\, 2^{\,i-22}, \qquad R = K_1 - C_2\,\delta, \qquad y = K_0 - R\,\delta .$$

**Savings.**
- Because every digit is $\pm 1$, each Horner row is a plain addition or subtraction rather than a controlled multiplication.
- The derivative register is split into a short residual and coarse bits, the constant high part of the result is phased early, and the coefficient registers are erased by measurement with one merged phase repair.
- The lowest accumulator bit is kept virtual and its carry is formed algebraically; the derivative's lowest wire is removed by refitting the parity of $K_1$ per segment.
- The eleven narrowest outer rows add into part of the low word plus a two-bit counter that is folded in once, and the two full-width rows start with a one-bit ripple prefix.
- Normalization uses a ladder of one-bit stages with a refitted margin.
- The output phase is fused: shifts by one and two positions are applied with controlled swaps, and each rotation reads its bit through a single temporary.
- Gates whose outcome is fixed on every legal input are replaced by Clifford operations, and eligible erasing Toffolis become an X-basis measurement followed by a conditioned CZ.
- Phase terms that are constant on the masked input domain are dropped, rotations whose angle allows it are moved to Clifford positions, and constant curvature blocks are folded.
- Further qubits are removed by reusing wires whose values are known on the legal domain; at this width one qubit is worth about 22 Toffolis, so the circuit accepts a few more Toffolis to save them.
- Lookup output bits that are Boolean functions of other outputs are recomputed when used instead of being stored, thirty carry prefixes are finished and erased early, five increment-tail workspaces are removed by temporarily erasing the top accumulator bit, the second fused phase pass alternates between two buffers, and selected input bits and normalization flags are parked in freed slots and restored with one Toffoli each. The circuit has 68 qubits; at this width one qubit is worth about 1.5% of the cost.

**Approximation.** The quadratic coefficients use short precisions and are fitted exactly on the reachable inputs. The 44 rotation angles are calibrated over the whole input domain; the worst phase error reaches 0.9998 of the tolerance after rotation synthesis, so almost the whole budget is spent. The verifier tests every legal input 64 times.

## 2. DQI decoder

**Role.** Decoded Quantum Interferometry turns an optimization problem over a finite field into the decoding of a Reed–Solomon code. Its dominant cost is a reversible decoder that runs in superposition.

**Computation.** The decoder receives 70 syndrome symbols over GF($2^{12}$) and a classical record $c$. It applies $(-1)^{\langle c, e\rangle}$ for the error $e$ of weight at most 35 and returns all workspace to zero. The locator $\sigma$ and evaluator $\Omega$ solve

$$S(z)\,\sigma(z) \equiv \Omega(z) \pmod{z^{70}}.$$

**Savings.**
- The 70 syndrome words are rewritten in place as the partial quotients of a continued fraction of $S(z)$, so the decoder needs almost no workspace beyond its 840 input qubits.
- Consecutive levels share one power-series reciprocal, so each pair of partial quotients costs a single inversion.
- The fraction is converted in place into $\sigma$ and $\Omega$, a Chien/Forney sweep applies the phase under the record bits $c_j$, and both transformations then run backwards.
- Seventy classical field words $\lambda_i$, solved from a Vandermonde system, absorb the record at positions 0–69 into a known syndrome phase $\mathrm{Tr}(\sum_i \lambda_i s_i)$, so the sweep skips those positions.
- The extraction buffer is released right after its measurement, the Forney evaluation keeps the denominator's constant coefficient implicit, and the gap history is stored as two short pointers.
- The tower-field norm of an inverse is the inverse of the norm, so the norm word is recycled instead of stored, and pairs of field products share their matrix schedule.
- For each extraction step the circuit chooses between single, paired and matrix-paired reciprocals. The choice comes from a dynamic program over the measured Toffoli and qubit cost of every option, and the conditional scaler is reused between pairs.

**Approximation.** A missing leading coefficient is repaired by a paired 3+1 step, and every history with at most two gaps is represented exactly. Rarer patterns are not handled. On the competition distribution they occur too seldom to raise $R(\varepsilon_+)$ above its floor, which the verifier sets from a Clopper–Pearson bound over 102,400 shots.

## 3. FeMoCo angle lookups

**Role.** FeMoCo is the active site of nitrogenase. Its block encoding is the core of a quantum chemistry algorithm for nitrogen fixation. The lookup circuits stream the 75 Givens-rotation angles of every factor vector into word registers.

**Factorization.** The two-body tensor is written with four groups of 303 unit vectors and 128 weight columns, $(R, B, C) = (4, 303, 128)$, and every angle is stored with **8 bits**. The short words are the main lever: the organizers' state-preparation cost model sizes its temporary workspace by the word width, so the whole block encoding falls from about 740 to 448 qubits, while the Toffoli count rises only modestly. The symmetry shift, the one-body term and the Givens tree are the organizers' own; the Hamiltonian stays physical. Direct 8-bit rounding of a good 10-bit fit is far outside the error budget; accuracy is recovered by constrained discrete refits of the angle words with transport of the coefficients and a larger dictionary, at the price of a slightly higher normalization. With the angle words, and therefore every lookup circuit, held fixed, the weights and identity offsets of all four groups are then refit jointly against the complete Hamiltonian, which lowers $\lambda_\text{eff}$ from 49.53 to 48.75.

**Circuits.** The 1,288 vectors are encoded on the log-depth tree in the signed half-open gauge, and fifteen exact lookup circuits stream them, each within 128 qubits. Each lookup is compiled from the algebraic normal form of its table with four-address difference banks; every legal address is checked exactly.

**Approximation.** The factorization spends the accuracy budget almost fully: the one-body error is 15.98 mHa and the two-body error 19.9998 mHa, against limits of 20 mHa, at $\lambda_\text{eff} = 48.75$, with $E_\text{gap}$ just above its minimum of 3.914898 Ha. The halved qubit count more than pays for the higher normalization.

## 4. Results

All values come from the official verifiers at full shot counts in this run. The decoder's failure count and the mean Toffoli counts depend slightly on the verification seed.

In [ ]:
r = RESULTS
rows = [
    ('Inverse square root', r['invsqrt']['num_qubits'], f"{r['invsqrt']['toffoli_count']:,.0f}",
     f"C = {r['invsqrt']['toffoli_equivalent']:,.0f} ({r['invsqrt']['rotation_count']:.0f} rotations)", 'invsqrt'),
    ('DQI decoder', r['dqi']['num_qubits'], f"{r['dqi']['toffoli_count']:,.0f}",
     f"R = {r['dqi']['repetition_factor']:.3f} ({r['dqi']['value_errors'] + r['dqi']['phase_errors']} failed of {r['dqi']['verified_shots']:,})", 'dqi'),
    ('FeMoCo lookups', r['dfthcsos']['num_qubits'], f"{r['dfthcsos']['toffoli_count']:,.0f}",
     f"λ_eff = {r['dfthcsos']['lambda_eff']:.2f}", 'dfthcsos'),
]
table = pd.DataFrame([{'Task': name, 'Qubits': q, 'Toffoli': t, 'Other': other,
                       'Cost': f"{r[k]['score']:.4g}", 'Reference cost': f'{REFERENCE[k]:.4g}',
                       'Reference / cost': f"{REFERENCE[k] / r[k]['score']:.2f}"} for name, q, t, other, k in rows])
score = math.prod(REFERENCE[k] / r[k]['score'] for k in REFERENCE) ** (1 / 3)
display(table.style.hide(axis='index'))
print(f'Score (geometric mean of reference / cost): {score:.3f}')

## Sources and reproducibility

The notebook runs on a CPU with the competition data and the attached dataset. The decoder sources are in our dataset folder `circuit_sources/`; the inverse-square-root and chemistry circuits are unchanged copies of mrmeows' open circuits, stored in the dataset folder `open_circuits/` with their licence and checked against SHA-256 values written in the code cell. The code cell builds the decoder, assembles the files, runs the official verifiers and writes `submission.zip` only after all three pass.

- Organizers: task definitions, verifiers, kickmix and the log-depth Givens tree ([competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).
- Inverse square root: the circuit descends from Phuc Thai's EXP-222/226a line ([dataset](https://www.kaggle.com/datasets/phucthaiv02/qcc-best-invsqrt-circuit)). Its construction extends the 84-qubit circuit of this notebook's earlier versions, which builds on [Sumit Chongder](https://www.kaggle.com/code/sumitchongder/notebook2f196e646b) (signed digits), [Rudra](https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing) (derivative split), [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation) and [kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation) (exact-domain rewrites), [zongaobian](https://www.kaggle.com/code/zongaobian/quantum-circuits-signed-digits-bm-psi-qrom) (virtual low bit) and [lightningv08](https://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra) / [Mizero](https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char) (calibration). Phuc Thai added the carry-save rows, the parity refit, the normalization ladder, the measurement-plus-CZ rewrite and the domain phase mask (EXP-226a); the fused phase stage is due to [Marsyas](https://www.kaggle.com/code/marsyas/vqcc-invsqrt-13-94x-full-source). The 78-qubit stage is [busyaprime's](https://www.kaggle.com/code/busyaprime/33-lb-11-94676-78-qubit-invsqrt-all-verified) seven verified edits of Phuc Thai's EXP222, and the 68-qubit circuit used here is [mrmeows'](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits) further storage and uncomputation edits of it (both Apache 2.0).
- DQI: in-place continued fraction, gap repair and arithmetic by [somejoint](https://www.kaggle.com/code/somejoint/quantum-circuits-8-95-lb-cost-breakdown); classical-mask projection by [manwithacat](https://www.kaggle.com/code/manwithacat/dqi-classical-mask-projection-9-13909); paired extraction by [Phuc Thai](https://www.kaggle.com/code/phucthaiv02/qcc-verified-portfolio-df-thc-retraining-study) and [ahmedberatozer](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression); norm recycling and paired matrix schedule by [anazemcev](https://www.kaggle.com/code/anazemcev/quantum-circuits-thc-joint-norm-881q-dqi). The decoder base is somejoint's source; the cost-driven extraction schedule and the shared scaler are ours. The reversible decoder line goes back to A. Gu et al., [arXiv:2610.22367](https://arxiv.org/abs/2610.22367).
- FeMoCo: the 8-bit $(4, 303, 128)$ factorization and its fifteen lookup circuits are by [mrmeows](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits) (Apache 2.0), with four-address lookup ideas from marculera; the joint refit of its weights and offsets is ours; the lineage includes the single-group factorization and algebraic-normal-form compiler of this notebook's earlier versions, ahmedberatozer (penalized fit, sensitivity-ordered quantization), anazemcev (joint refit), Artem1410 and kozykappa (lookup and erasure) and Sumit Chongder (banked lookups).
- C. Gidney, [Halving the cost of quantum addition](https://quantum-journal.org/papers/q-2018-06-18-74/) (2018); G. H. Low et al., [PRX 15, 041016](https://journals.aps.org/prx/abstract/10.1103/pb2g-j9cw) (2025).

All reused notebook code is Apache 2.0 (Kaggle's licence for public notebooks unless stated otherwise); this notebook and its dataset are released under Apache 2.0.

In [ ]:
P.write_zip(SUB, WORK / 'submission.zip')